# Condition Comparison: Significance Analysis

Compare probe runs along any axis (layer, probe model, GNN backbone, RMSD
threshold, ...) with paired bootstrap tests and Holm-Bonferroni correction.

Runs are **discovered, not constructed**: `find_probe_runs()` in
`prob.paths_and_io` is the read-side inverse of `get_out_dir()` / `get_exp_dirs()`,
so this notebook never restates the directory layout or checks paths for
existence. `attach_run_metrics()` folds in what each run already wrote to its
`reports/<probe>_summary.json`, and `load_run_predictions()` reads the
`y_true,y_pred` CSV for the rows that survive filtering.

Bootstrapped metrics come from `prob.prob_stats.METRIC_FNS` (r2, rmse, mae,
pearson); every table and printout below is generated from that registry, so
adding a metric there flows through this notebook with no edits here.

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
from pathlib import Path

import numpy as np
import pandas as pd

# Walk up to the repo root (same marker logic as paths_and_io._find_root) so the
# notebook does not depend on how deep it sits, then import prob.* from there.
ROOT = Path(os.environ.get("HOME_PROJ_DIR") or next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "prob" / "paths_and_io.py").exists()
))
os.chdir(ROOT)
pd.set_option("display.width", 200)
print(f"Working directory: {ROOT}")

Working directory: /home/fatemeh/thesis/kinodata-3D-affinity-prediction


In [2]:
from prob.paths_and_io import find_probe_runs, attach_run_metrics, load_run_predictions
from prob.prob_stats import METRIC_FNS, compare_two_conditions, compare_multiple_conditions
from prob.prob_plots import plot_conditions_box

print("Imports successful! Bootstrapped metrics:", list(METRIC_FNS))

Imports successful! Bootstrapped metrics: ['r2', 'rmse', 'mae', 'pearson']


## Configuration

The slice to compare. `AXIS` is the column whose values become the conditions;
everything else is held fixed by the filters. Set `LAYERS = None` (or drop any
other filter) to take whatever actually ran.

In [3]:
# CONFIGURATION: the fixed slice ...

slice = dict(gnn_model_type="CGNN-3D", 
             rmsd_threshold=2, 
             split_type="random-k-fold", 
             target="affinity", 
             prob_model="mlp")

# ... and the axis being compared. Any decoded column works: "layer",
# "prob_model", "gnn_model_type", "rmsd_threshold". Widen the matching filter
# above (e.g. PROBE = ["ridge", "mlp"]) when you switch axis.
AXIS = "layer"
RANDOM_STATE = 96

## What actually ran

One row per (gnn, rmsd, split, target, probe, layer) experiment with a saved
predictions CSV. Baselines (`<target>_shuffled_ident`) come back in the same
frame, flagged by `is_baseline`, so they need no second lookup.

Call `find_probe_runs()` with no arguments to see the whole sweep first -- that
is also the honest answer to "which cells of the design are still missing".

In [4]:
runs = attach_run_metrics(find_probe_runs(**slice))

if runs.empty:
    raise SystemExit("No runs matched -- relax a filter, or call find_probe_runs() bare to see the sweep.")

metric_cols = [c for c in ("r2", "rmse", "mae", "n_test_samples") if c in runs.columns]
runs[["target_full", "is_baseline", "prob_model", "layer", *metric_cols]]

,target_full,is_baseline,prob_model,layer,r2,rmse,mae,n_test_samples
0,affinity,False,mlp,1,0.183490,1.188194,0.965087,4124
1,affinity,False,mlp,2,0.288415,1.109225,0.888760,4124
2,affinity,False,mlp,3,0.397945,1.020292,0.794451,4124
3,affinity_shuffled_ident,True,mlp,1,-0.011336,1.311594,1.071297,4124
4,affinity_shuffled_ident,True,mlp,2,-0.018352,1.316135,1.078860,4124
5,affinity_shuffled_ident,True,mlp,3,-0.038705,1.329222,1.091698,4124


## Load predictions

Conditions are keyed by the `AXIS` value. Paired tests require the same test
samples in the same order, which holds across layers/probes of one target; the
`shuffled_ident` baselines deliberately permute the ident->target assignment, so
their `y_true` differs and they cannot enter the paired tests. They are kept
aside for the distribution plots at the end.

In [ ]:
def _conditions(frame, prefix=""):
    """{condition_name: (y_true, y_pred)} keyed by the AXIS value."""
    out = {}
    for run in frame.sort_values(AXIS).itertuples():
        key = f"{prefix}{AXIS}_{getattr(run, AXIS)}"
        if key in out:
            raise ValueError(f"Duplicate condition {key!r}: {AXIS} does not identify a run in this slice.")
        out[key] = load_run_predictions(run)
    return out


predictions = _conditions(runs[~runs.is_baseline])
baseline_predictions = _conditions(runs[runs.is_baseline], prefix="shuffled_")

for name, (y_true, _) in predictions.items():
    print(f"OK {name}: {len(y_true)} samples")
print(f"\nLoaded {len(predictions)} conditions and {len(baseline_predictions)} baselines")

## Pairwise Comparison (Two Conditions)

The first two conditions in the slice, with the full statistics for every
bootstrapped metric.

In [ ]:
if len(predictions) >= 2:
    (name_a, (yt_a, yp_a)), (name_b, (yt_b, yp_b)) = list(predictions.items())[:2]
    result = compare_two_conditions(
        name_a, yt_a, yp_a,
        name_b, yt_b, yp_b,
        random_state=RANDOM_STATE,
    )

    print("=" * 70)
    print(f"{name_a} vs {name_b}".upper())
    print("=" * 70)
    print(f"\nSample size: {result['n_samples']}")
    for metric in METRIC_FNS:
        print(f"\n{metric.upper()} difference ({name_a} - {name_b}):")
        print(f"  Point estimate:      {result[f'delta_{metric}']:+.4f}")
        print(f"  95% CI:              [{result[f'delta_{metric}_ci_low']:+.4f}, "
              f"{result[f'delta_{metric}_ci_high']:+.4f}]")
        print(f"  Bootstrap p-value:   {result[f'delta_{metric}_bootstrap_pval']:.4f}")

    print("\nWilcoxon signed-rank test (squared errors):")
    print(f"  p-value:               {result['wilcoxon_pval']:.4f}")
    print(f"  Median d(squared err): {result['median_delta_squared_error']:+.4f}")
else:
    print("Not enough conditions loaded to compare")

## Multiple Comparisons (All Pairs)

All pairs at once, with Holm-Bonferroni correction applied per p-value column.

In [ ]:
if len(predictions) >= 2:
    comparison_df = compare_multiple_conditions(predictions, random_state=RANDOM_STATE)

    print(f"\nAll pairwise comparisons ({len(comparison_df)} pairs):")
    print("=" * 100)
    display_cols = ["condition_a", "condition_b"]
    for metric in METRIC_FNS:
        display_cols += [f"delta_{metric}", f"delta_{metric}_bootstrap_pval_holm"]
    display_cols += ["wilcoxon_pval", "wilcoxon_pval_holm"]
    print(comparison_df[display_cols].to_string(index=False))
else:
    print("Need at least 2 conditions to compare")

## Summary with Significance Markers

In [ ]:
def _stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"


if len(predictions) >= 2:
    print("\nSummary of Holm-Bonferroni corrected results:")
    print("-" * 80)

    for _, row in comparison_df.iterrows():
        print(f"\n{row['condition_a']} vs {row['condition_b']}:")
        for metric in METRIC_FNS:
            print(f"  d{metric} = {row[f'delta_{metric}']:+.4f} "
                  f"[{row[f'delta_{metric}_ci_low']:+.4f}, {row[f'delta_{metric}_ci_high']:+.4f}] "
                  f"{_stars(row[f'delta_{metric}_bootstrap_pval_holm'])}")
        print(f"  Wilcoxon p_holm = {row['wilcoxon_pval_holm']:.4f} {_stars(row['wilcoxon_pval_holm'])}")

    print("\nSignificance codes: *** p<0.001  ** p<0.01  * p<0.05  ns=not significant")
else:
    print("Need at least 2 conditions to compare")

## Save Results

Written next to the runs being compared -- `exp_root` is
`.../<target>/<probe>/<layer>`, so its parent is the probe directory shared by
every condition on the layer axis.

In [ ]:
if len(predictions) >= 2:
    probe_runs = runs[~runs.is_baseline]
    output_file = Path(probe_runs.iloc[0]["exp_root"]).parent / f"{PROBE}_{AXIS}_comparison.csv"
    output_file.parent.mkdir(parents=True, exist_ok=True)
    comparison_df.to_csv(output_file, index=False)

    print(f"OK saved to: {output_file}")
    print(f"\nDataFrame shape: {comparison_df.shape}")
    print(f"Columns: {', '.join(comparison_df.columns)}")
else:
    print("Cannot save: need at least 2 conditions")

## Distribution Visualization

Per-sample metric distributions across conditions, with the `shuffled_ident`
baselines loaded above as the reference. Figures are written beside the CSV.

In [7]:
runs.iloc[0]

gnn_model_type                                                CGNN-3D
rmsd_threshold                                                    2.0
split_type                                              random-k-fold
target                                                       affinity
target_full                                                  affinity
is_baseline                                                     False
prob_model                                                        mlp
layer                                                               1
exp_root            /home/fatemeh/thesis/kinodata-3D-affinity-pred...
predictions_path    /home/fatemeh/thesis/kinodata-3D-affinity-pred...
summary_path        /home/fatemeh/thesis/kinodata-3D-affinity-pred...
figures_dir         /home/fatemeh/thesis/kinodata-3D-affinity-pred...
r2                                                            0.18349
rmse                                                         1.188194
mae                 

In [9]:
FIG_DIR = Path(runs[~runs.is_baseline].iloc[0]["exp_root"]).parent

if predictions and baseline_predictions:
    for metric, label in [("squared_error", "Squared Error"), ("abs_residual", "Absolute Residual")]:
        plot_conditions_box(
            predictions,
            baseline_dict=baseline_predictions,
            metric=metric,
            title=f"{TARGET} ({PROBE}): {label} Distribution",
            save_path=FIG_DIR / f"{PROBE}_{metric}_box",
            show=True,
        )
        print(f"OK box plot ({metric}) saved under {FIG_DIR / 'figures'}")
else:
    print("Need both conditions and baselines for the box plots")

NameError: name 'baseline_predictions' is not defined